# Positions lab: RoPE by hand

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ankitrgupta1/silicon-to-agents/blob/main/labs/positions.ipynb)

See why rotary position embeddings make attention depend on *relative* position. Companion to [Embeddings & Positions](../03-foundations/embeddings-positions.md). Runs on CPU.

In [ ]:
try:
    import s2a
except ImportError:
    %pip install -q git+https://github.com/ankitrgupta1/silicon-to-agents

import numpy as np
import matplotlib.pyplot as plt
from s2a.positions import apply_rope, rope_frequencies, sinusoidal_positions

## 1. Attention without positions ignores order

Shuffle the tokens and the attention scores just get shuffled too.

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(4, 16))           # 4 tokens
perm = [2, 0, 3, 1]
scores = X @ X.T
scores_shuffled = X[perm] @ X[perm].T
print("Same scores, just reordered:", np.allclose(scores_shuffled, scores[np.ix_(perm, perm)]))

## 2. RoPE: only the distance matters

Put a query at position `m` and a key at position `n`. The score is the same for every pair with the same offset `m - n`.

In [ ]:
q, k = rng.normal(size=(1, 64)), rng.normal(size=(1, 64))

def score(m, n):
    return (apply_rope(q, [m]) @ apply_rope(k, [n]).T).item()

for m, n in [(10, 7), (110, 107), (5000, 4997), (10, 6)]:
    print(f"query at {m:>5}, key at {n:>5}, offset {m - n}: score {score(m, n):+.4f}")

## 3. Score versus distance

Use the same vector as query and key, so the score at distance 0 is as high as possible. As distance grows, the rotations drift apart and the score tends to fall, giving attention a built-in (noisy) preference for nearby tokens.

In [ ]:
v = np.ones((1, 128))                   # an all-ones vector shows the effect cleanly
dists = np.arange(0, 2000)
for base in [10_000, 500_000]:
    s = [(apply_rope(v, [0], base) @ apply_rope(v, [d], base).T).item() for d in dists]
    plt.plot(dists, s, label=f"base {base:,}")
plt.xlabel("distance between query and key"); plt.ylabel("score"); plt.legend(); plt.show()

## 4. Fast and slow pairs

Each dimension pair rotates at its own speed. Fast pairs track local order; slow pairs tell far-apart positions apart. A bigger base makes the slow pairs slower.

In [ ]:
for base in [10_000, 500_000]:
    wavelength = 2 * np.pi / rope_frequencies(128, base)
    plt.semilogy(wavelength, label=f"base {base:,}")
plt.xlabel("dimension pair"); plt.ylabel("positions per full turn (log scale)"); plt.legend(); plt.show()

## 5. The original sinusoidal encodings

Same frequencies, but added to the embeddings instead of rotating queries and keys.

In [ ]:
plt.imshow(sinusoidal_positions(100, 64).T, aspect="auto", cmap="RdBu")
plt.xlabel("position"); plt.ylabel("dimension"); plt.colorbar(); plt.show()

## Try it

- In section 3, try a random vector instead of all ones. Is the decay still there? Why is it noisier?
- Implement position interpolation: call `apply_rope` with `positions / 4`. How does the score-versus-distance curve change?
- Check that `apply_rope` never changes a vector's length. Why does that matter?